# 🛠️ Notebook 2 · Hotel Management — Implementation

In Notebook 1 we picked the classes. Now we implement the **booking engine**
and grow it from a naive version into something closer to a real hotel system.

We'll add, one step at a time:
1. A correct overlap check (the heart of any reservation system).
2. A reservation **status** (ACTIVE / CANCELLED / CHECKED_IN / CHECKED_OUT).
3. A **24-hour cancellation policy** (full refund vs. no refund).
4. A **housekeeping log** and room availability vs. cleanliness.
5. A tiny **invoice** with room service charges.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/hotel-management
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If missing: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ Shared domain types

We reuse the clean classes from Notebook 1 and add a `ReservationStatus` enum.
Enums beat raw strings because they are **typo-proof** and **auto-documenting**.


In [ ]:
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta
from enum import Enum
import itertools

class RoomType(Enum):
    SINGLE = 100
    DOUBLE = 150
    SUITE  = 300

class ReservationStatus(Enum):
    ACTIVE       = "ACTIVE"
    CHECKED_IN   = "CHECKED_IN"
    CHECKED_OUT  = "CHECKED_OUT"
    CANCELLED    = "CANCELLED"

class RoomStatus(Enum):
    AVAILABLE   = "AVAILABLE"
    OCCUPIED    = "OCCUPIED"
    NEEDS_CLEAN = "NEEDS_CLEAN"
    MAINTENANCE = "MAINTENANCE"

@dataclass(frozen=True)
class Room:
    number: int
    type: RoomType

@dataclass(frozen=True)
class Guest:
    id: str
    name: str
    email: str = ""


## 2️⃣ The overlap check — tiny but critical

Two date ranges `[a_in, a_out)` and `[b_in, b_out)` **overlap** iff
`a_in < b_out AND b_in < a_out`. This one line catches every case:
fully inside, fully outside, touching at an edge, and partial overlap.

> We treat intervals as **half-open** — checkout day is free for a new guest.
> That matches how real hotels operate and avoids off-by-one pain.


In [ ]:
def overlaps(a_in, a_out, b_in, b_out) -> bool:
    return a_in < b_out and b_in < a_out

# Sanity checks you can eyeball.
d = date
assert     overlaps(d(2026,5,1), d(2026,5,4), d(2026,5,3), d(2026,5,5))   # partial
assert     overlaps(d(2026,5,1), d(2026,5,10), d(2026,5,2), d(2026,5,5))  # inside
assert not overlaps(d(2026,5,1), d(2026,5,4), d(2026,5,4), d(2026,5,6))   # edge-touch OK
assert not overlaps(d(2026,5,1), d(2026,5,4), d(2026,5,5), d(2026,5,6))   # separate
print("overlap tests pass ✅")


## 3️⃣ Reservation — data + small behaviours

Computed properties (`nights`, `total`) keep callers from recomputing the same
math three different ways. `can_full_refund` encodes the 24-hour policy in
exactly one place — the `Hotel` will just *ask*.


In [ ]:
@dataclass
class Reservation:
    id: int
    room: Room
    guest: Guest
    check_in: date
    check_out: date
    status: ReservationStatus = ReservationStatus.ACTIVE
    extras: list = field(default_factory=list)  # room service / amenities

    @property
    def nights(self) -> int:
        return (self.check_out - self.check_in).days

    @property
    def room_total(self) -> int:
        return self.nights * self.room.type.value

    @property
    def extras_total(self) -> int:
        return sum(amount for _, amount in self.extras)

    @property
    def total(self) -> int:
        return self.room_total + self.extras_total

    def can_full_refund(self, now: date) -> bool:
        # Requirement: full refund if cancelled > 24h before check-in.
        return (self.check_in - now) > timedelta(days=1)

    def invoice_lines(self):
        yield (f"Room {self.room.number} ({self.room.type.name}) × {self.nights} nights",
               self.room_total)
        for label, amount in self.extras:
            yield (label, amount)


## 4️⃣ Attempt A — a naive `Hotel` (bad)

Everything in one method, silent bugs, no status, no housekeeping. We'll keep
this around only to contrast with the good version.


In [ ]:
class HotelNaive:
    def __init__(self, rooms):
        self.rooms = rooms
        self.reservations = []

    def book(self, room_number, guest, ci, co):
        # 🔴 no date validation, 🔴 no status tracking, 🔴 returns a dict not an object
        self.reservations.append({"room": room_number, "guest": guest.name, "in": ci, "out": co})

naive = HotelNaive([Room(101, RoomType.SINGLE)])
naive.book(101, Guest("G1","Ada"), date(2026,5,4), date(2026,5,1))  # 😱 negative nights, accepted
print("accepted a bad booking:", naive.reservations)


## 5️⃣ Attempt B — the good `Hotel` (best)

A single class owns:
- the **rooms collection** and **reservations ledger**,
- **availability** (using our `overlaps`),
- **status transitions** (check-in, check-out, cancel),
- a **housekeeping log** so we don't book a dirty room for same-day turnover.

Notice how short each method stays — that's the hallmark of good responsibility splits.


In [ ]:
@dataclass
class HousekeepingTask:
    room_number: int
    at: datetime
    staff: str
    done: bool = False

class Hotel:
    def __init__(self, rooms):
        # Per-INSTANCE id counter. As a class attribute (`Hotel._rid = count(1)`)
        # two hotels would share one sequence, so hotel B's first reservation
        # would be #7 — and re-running this cell would keep drifting.
        self._rid = itertools.count(1)
        self.rooms = {r.number: r for r in rooms}
        self.reservations: list[Reservation] = []
        self.room_status = {r.number: RoomStatus.AVAILABLE for r in rooms}
        self.housekeeping: list[HousekeepingTask] = []

    # ---- availability --------------------------------------------------
    # Physical states that make a room unsellable no matter what the ledger says.
    BLOCKING_STATUSES = (RoomStatus.MAINTENANCE, RoomStatus.NEEDS_CLEAN)

    def is_available(self, room_number, ci, co) -> bool:
        # Two INDEPENDENT reasons a room can be unavailable, and we must check both:
        #   1. physical condition of the room right now (dirty / under repair)
        #   2. the reservation ledger (someone else already holds those dates)
        if self.room_status[room_number] in self.BLOCKING_STATUSES:
            return False
        for r in self.reservations:
            if r.status == ReservationStatus.CANCELLED or r.room.number != room_number:
                continue
            if overlaps(ci, co, r.check_in, r.check_out):
                return False
        return True

    def search(self, room_type, ci, co) -> list[Room]:
        return [r for r in self.rooms.values()
                if r.type == room_type and self.is_available(r.number, ci, co)]

    # ---- bookings ------------------------------------------------------
    def reserve(self, room_number, guest, ci, co) -> Reservation:
        if co <= ci:
            raise ValueError("check_out must be after check_in")
        if room_number not in self.rooms:
            raise KeyError(f"unknown room {room_number}")
        if not self.is_available(room_number, ci, co):
            raise RuntimeError(f"room {room_number} not available for {ci}→{co}")
        res = Reservation(next(self._rid), self.rooms[room_number], guest, ci, co)
        self.reservations.append(res)
        return res

    def cancel(self, reservation: Reservation, *, today: date) -> int:
        if reservation.status != ReservationStatus.ACTIVE:
            raise RuntimeError(f"cannot cancel from status {reservation.status.name}")
        refund = reservation.total if reservation.can_full_refund(today) else 0
        reservation.status = ReservationStatus.CANCELLED
        return refund

    # ---- lifecycle -----------------------------------------------------
    def check_in(self, reservation: Reservation):
        if reservation.status != ReservationStatus.ACTIVE:
            raise RuntimeError("only ACTIVE reservations can check in")
        reservation.status = ReservationStatus.CHECKED_IN
        self.room_status[reservation.room.number] = RoomStatus.OCCUPIED

    def check_out(self, reservation: Reservation, *, staff: str = "auto"):
        if reservation.status != ReservationStatus.CHECKED_IN:
            raise RuntimeError("guest must be checked in first")
        reservation.status = ReservationStatus.CHECKED_OUT
        self.room_status[reservation.room.number] = RoomStatus.NEEDS_CLEAN
        self.housekeeping.append(
            HousekeepingTask(reservation.room.number, datetime.now(), staff)
        )

    def complete_housekeeping(self, room_number: int):
        for t in self.housekeeping:
            if t.room_number == room_number and not t.done:
                t.done = True
        self.room_status[room_number] = RoomStatus.AVAILABLE

    # ---- billing -------------------------------------------------------
    def add_room_service(self, reservation: Reservation, label: str, amount: int):
        if reservation.status not in (ReservationStatus.ACTIVE, ReservationStatus.CHECKED_IN):
            raise RuntimeError("cannot charge a closed reservation")
        reservation.extras.append((label, amount))

    def print_invoice(self, reservation: Reservation):
        print(f"Invoice #{reservation.id} for {reservation.guest.name}")
        print("-" * 44)
        for label, amount in reservation.invoice_lines():
            print(f"  {label:<34}${amount:>6}")
        print("-" * 44)
        print(f"  {'TOTAL':<34}${reservation.total:>6}")

## 6️⃣ Walk-through — a realistic day at the hotel 🏨

Booking, overlap rejection, room-service charges, check-in / check-out,
housekeeping, and two kinds of cancellation (full refund vs. no refund).


In [ ]:
hotel = Hotel([
    Room(101, RoomType.SINGLE),
    Room(102, RoomType.SINGLE),
    Room(201, RoomType.DOUBLE),
    Room(301, RoomType.SUITE),
])

ada   = Guest("G1", "Ada",   "ada@ex.com")
grace = Guest("G2", "Grace", "grace@ex.com")
linus = Guest("G3", "Linus", "linus@ex.com")

# --- search & reserve ---
print("singles free May 1-4:", [r.number for r in hotel.search(RoomType.SINGLE, date(2026,5,1), date(2026,5,4))])
r1 = hotel.reserve(101, ada,   date(2026,5,1), date(2026,5,4))
r2 = hotel.reserve(201, grace, date(2026,5,2), date(2026,5,6))
print(f"booked #{r1.id} {r1.guest.name} → ${r1.total}")
print(f"booked #{r2.id} {r2.guest.name} → ${r2.total}")

# --- overlap rejection ---
try:
    hotel.reserve(101, linus, date(2026,5,3), date(2026,5,5))
except RuntimeError as e:
    print("overlap blocked ✅:", e)

# --- non-overlapping on same room is fine ---
r3 = hotel.reserve(101, linus, date(2026,5,4), date(2026,5,6))
print(f"booked #{r3.id} {r3.guest.name} (back-to-back)")


In [ ]:
# --- lifecycle + room service + invoice ---
hotel.check_in(r1)
hotel.add_room_service(r1, "Breakfast x3", 45)
hotel.add_room_service(r1, "Mini-bar",     18)
hotel.check_out(r1, staff="Maria")
hotel.print_invoice(r1)

print("room 101 status after checkout:", hotel.room_status[101].name)

# A dirty room is NOT sellable, even though the ledger shows those dates free.
# This is why RoomStatus exists alongside the reservation list.
print("101 bookable while dirty? ", hotel.is_available(101, date(2026,8,1), date(2026,8,3)))
try:
    hotel.reserve(101, linus, date(2026,8,1), date(2026,8,3))
except RuntimeError as e:
    print("dirty-room booking blocked ✅:", e)

hotel.complete_housekeeping(101)
print("room 101 status after cleaning:", hotel.room_status[101].name)
print("101 bookable once clean?  ", hotel.is_available(101, date(2026,8,1), date(2026,8,3)))

In [ ]:
# --- cancellation policy ---
late  = hotel.reserve(301, ada,   date(2026,6,1), date(2026,6,3))   # plenty of time
rush  = hotel.reserve(102, grace, date(2026,5,5), date(2026,5,7))   # checking in tomorrow

refund_full = hotel.cancel(late, today=date(2026,5,20))
refund_none = hotel.cancel(rush, today=date(2026,5,4))  # <24h before check-in
print(f"late cancel refund: ${refund_full}  (full refund)")
print(f"rush cancel refund: ${refund_none} (forfeit — inside 24h window)")


## 6️⃣b Verify the design — assertions, not just prints

The walk-through above *prints* the happy path. Prints only prove the code ran; they prove
nothing about the rules. These assertions state the hotel's invariants in executable form, so
that a future refactor that breaks one fails immediately instead of quietly overbooking a room.

In [ ]:
check = Hotel([Room(1, RoomType.SINGLE), Room(2, RoomType.SUITE)])
g = Guest("GX", "Test")

# --- 1. Reservation ids restart per hotel (they are not global) ------------
other = Hotel([Room(1, RoomType.SINGLE)])
assert check.reserve(1, g, date(2026,1,1), date(2026,1,3)).id == 1
assert other.reserve(1, g, date(2026,1,1), date(2026,1,3)).id == 1, "ids must be per-hotel"

# --- 2. Date validation: zero- and negative-length stays are rejected ------
for ci, co in [(date(2026,3,4), date(2026,3,4)),      # zero nights
               (date(2026,3,4), date(2026,3,1))]:     # backwards
    try:
        check.reserve(2, g, ci, co)
        raise AssertionError(f"{ci}->{co} should have been rejected")
    except ValueError:
        pass

# --- 3. Overlap: half-open intervals, so checkout day is immediately sellable
assert not check.is_available(1, date(2026,1,2), date(2026,1,4)), "partial overlap is busy"
assert     check.is_available(1, date(2026,1,3), date(2026,1,5)), "check-out day is free"

# --- 4. A cancelled reservation frees its dates ---------------------------
c = check.reserve(2, g, date(2026,2,1), date(2026,2,5))
assert not check.is_available(2, date(2026,2,2), date(2026,2,3))
check.cancel(c, today=date(2026,1,1))
assert check.is_available(2, date(2026,2,2), date(2026,2,3)), "cancelling releases the dates"
try:
    check.cancel(c, today=date(2026,1,1))                # already CANCELLED
    raise AssertionError("double-cancel must be rejected")
except RuntimeError:
    pass

# --- 5. Refund policy sits exactly on the 24-hour boundary ----------------
r = check.reserve(2, g, date(2026,4,10), date(2026,4,12))
assert     r.can_full_refund(date(2026,4,8)),  "2 days out -> full refund"
assert not r.can_full_refund(date(2026,4,9)),  "exactly 24h out -> no refund"
assert not r.can_full_refund(date(2026,4,10)), "same day -> no refund"

# --- 6. Lifecycle is a state machine, not a free-for-all -----------------
r2 = check.reserve(1, g, date(2026,5,1), date(2026,5,3))
try:
    check.check_out(r2)                                  # never checked in
    raise AssertionError("cannot check out before checking in")
except RuntimeError:
    pass
check.check_in(r2)
assert check.room_status[1] is RoomStatus.OCCUPIED
check.check_out(r2, staff="Ana")
assert check.room_status[1] is RoomStatus.NEEDS_CLEAN
assert not check.is_available(1, date(2026,9,1), date(2026,9,2)), "dirty room is unsellable"
check.complete_housekeeping(1)
assert check.room_status[1] is RoomStatus.AVAILABLE
assert check.is_available(1, date(2026,9,1), date(2026,9,2)),     "cleaning restores it"
assert all(t.done for t in check.housekeeping),                   "housekeeping log closed out"

# --- 7. Billing: totals are derived, never stored ------------------------
r3 = check.reserve(2, g, date(2026,6,1), date(2026,6,4))   # 3 nights x $300
assert r3.nights == 3 and r3.room_total == 900
check.add_room_service(r3, "Spa", 120)
assert r3.total == 1020 and r3.extras_total == 120
check.cancel(r3, today=date(2026,5,1))
try:
    check.add_room_service(r3, "Late charge", 50)          # closed reservation
    raise AssertionError("cannot charge a cancelled reservation")
except RuntimeError:
    pass

print("all hotel invariants hold ✅")

## 7️⃣ Putting the progression side by side

| Concern | Attempt 0 (dicts) | Attempt A (naive class) | Attempt B (clean) |
|---|---|---|---|
| Overlap detection | broken | missing | ✅ interval math |
| Type safety on room types | strings | strings | ✅ `Enum` |
| Reservation totals | ad-hoc | none | ✅ computed property |
| Status transitions | none | none | ✅ `ReservationStatus` |
| Cancellation policy | none | none | ✅ 24-hour rule |
| Housekeeping | none | none | ✅ log + `RoomStatus` |
| Billing | none | none | ✅ extras + invoice |

Each step was **small**, **behaviour-preserving** for existing callers, and
pushed rules into the class that *owns* the data.


## 🧠 Exercises
1. **Membership discount.** Add a `Guest.is_member` flag and give members 10%
   off the room total. Where does the discount belong — `Guest`, `Reservation`,
   or `Hotel`? Justify in one sentence.
2. **Find-any-room search.** Add `Hotel.search_any(ci, co)` that returns the
   cheapest available room across all types.
3. **Payment methods.** Introduce a `Payment` class with `method` in
   `{CARD, CASH, CHECK}` and record it on check-out. Reject checkout if unpaid.
4. **Multi-hotel chain.** What would change if one `HotelChain` owned many
   `Hotel` instances? (Hint: the ledger of reservations should probably stay
   per-hotel; search can aggregate.)

### Key takeaways
- Grow code **in small, named steps**; don't rewrite the world at once.
- Put rules next to the data they govern (overlap → `Hotel`, refund → `Reservation`).
- Use `Enum` for states and transitions — they're self-documenting and safe.
- Computed properties beat "keep this field in sync" every time.
